# Chapter 4 Lab — A Classical Baseline (The Thing to Beat)

**Think of it like this:** imagine teaching someone to sort laundry without ever explaining what a "shirt" or "sock" actually is. Instead, you hand them a measuring tape and a checklist: how big is it, how stretchy, does it have sleeves. They sort using only those numbers — no eyes needed, just measurements. That's exactly what we're about to do with sleep stages: instead of a human eyeballing the squiggle (Chapters 1–3), we'll extract a handful of numbers per epoch (band power from Chapter 3) and let an algorithm sort using only those.

A **Random Forest** is like asking a big group of slightly-different-opinion friends to each make a guess, then going with whatever the majority says. Each "friend" (a decision tree) looks at the measurements slightly differently, so their mistakes tend to cancel out.

**Goal of this lab:** build a real classifier, evaluate it properly, and get a confusion matrix you can actually read.

In [ ]:
!pip install -q mne scikit-learn
%matplotlib inline
import mne
import numpy as np
from mne.datasets.sleep_physionet.age import fetch_data

## The One Rule That Matters: Split By Subject

Before any code: we load **two different people** — Alice and Bob — and we train on one, test on the other. Never mix epochs from the same person across train and test; see the "Leaky or Legit?" game if you haven't played it yet. This single rule is the difference between a real result and a fake, inflated accuracy number.

In [ ]:
ALICE, BOB = 0, 1
[alice_files, bob_files] = fetch_data(subjects=[ALICE, BOB], recording=[1])

annotation_desc_2_event_id = {
    "Sleep stage W": 1, "Sleep stage 1": 2, "Sleep stage 2": 3,
    "Sleep stage 3": 4, "Sleep stage 4": 4, "Sleep stage R": 5,
}
event_id = {"Wake": 1, "N1": 2, "N2": 3, "N3": 4, "REM": 5}
id_to_name = {v: k for k, v in event_id.items()}

def load_epochs(files):
    raw = mne.io.read_raw_edf(files[0], stim_channel="Event marker", infer_types=True, preload=True, verbose="error")
    annot = mne.read_annotations(files[1])
    raw.set_annotations(annot, emit_warning=False)
    annot.crop(annot[1]["onset"] - 30 * 60, annot[-2]["onset"] + 30 * 60)
    raw.set_annotations(annot, emit_warning=False)
    events, _ = mne.events_from_annotations(raw, event_id=annotation_desc_2_event_id, chunk_duration=30.0)
    tmax = 30.0 - 1.0 / raw.info["sfreq"]
    return mne.Epochs(raw=raw, events=events, event_id=event_id, tmin=0.0, tmax=tmax, baseline=None, preload=True)

epochs_train = load_epochs(alice_files)
epochs_test = load_epochs(bob_files)
print("Train (Alice):", len(epochs_train), "epochs. Test (Bob):", len(epochs_test), "epochs.")

## Turn Each Epoch Into a Handful of Numbers

This is Chapter 3's spectrogram idea, condensed: for every 30-second epoch, compute how much delta, theta, alpha, sigma, and beta power it contains. Five numbers per epoch — the "measuring tape" from the laundry analogy.

In [ ]:
FREQ_BANDS = {"delta": [0.5, 4.5], "theta": [4.5, 8.5], "alpha": [8.5, 11.5], "sigma": [11.5, 15.5], "beta": [15.5, 30]}

def eeg_power_band(epochs):
    spectrum = epochs.compute_psd(picks="eeg", fmin=0.5, fmax=30.0, verbose="error")
    psds, freqs = spectrum.get_data(return_freqs=True)
    psds /= np.sum(psds, axis=-1, keepdims=True)
    X = []
    for fmin, fmax in FREQ_BANDS.values():
        psds_band = psds[:, :, (freqs >= fmin) & (freqs < fmax)].mean(axis=-1)
        X.append(psds_band.reshape(len(psds), -1))
    return np.concatenate(X, axis=1)

## Train the Random Forest

The pipeline below does the feature extraction AND classification in one step, so it works cleanly with scikit-learn's `.fit()` / `.predict()` pattern.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

pipe = make_pipeline(
    FunctionTransformer(eeg_power_band, validate=False),
    RandomForestClassifier(n_estimators=100, random_state=42),
)

y_train = epochs_train.events[:, 2]
y_test = epochs_test.events[:, 2]

pipe.fit(epochs_train, y_train)
y_pred = pipe.predict(epochs_test)

accuracy = (y_pred == y_test).mean()
print(f"Accuracy on Bob (never seen during training): {accuracy:.1%}")

## Read the Confusion Matrix

This is where the real insight lives — not the single accuracy number, but WHICH mistakes the model makes. Watch the N1/REM row and column especially.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

labels = sorted(event_id.values())
label_names = [id_to_name[l] for l in labels]
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(label_names)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar(im)
plt.title("Confusion Matrix (Bob, held out)")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=label_names))

## Done when...

You have a working baseline with a confusion matrix, and can point at where it fails — almost certainly N1 and REM confusion, same as every human scorer and every model in this course. That number you just got is **the thing Chapter 6's neural network has to actually beat**, not just match.